<a href="https://colab.research.google.com/github/alisony755/DS4400/blob/main/HW3/DS3000_HW3_Problem2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 2

In [17]:
# Import numpy, pandas, and sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [18]:
# Import Google Drive to upload data
from google.colab import drive
drive.mount('/content/drive')

# Read in data from file path
file_id = '1_hNHlID38Uf5lphkEyXVKz5_afFb1eve'
file_path = f"https://drive.google.com/uc?id={file_id}&export=download"
df_house = pd.read_csv(file_path)

# Drop unnecessary columns
df_house = df_house.drop(columns=['id', 'date', 'zipcode'])

# Print first few rows of data
df_house.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,47.6168,-122.045,1800,7503


In [19]:
# Convert regression problem into classification
# Houses above the median price -> class 1
# Houses below median price -> class 0

median_price = df_house["price"].median()

df_house["price_class"] = (df_house["price"] > median_price).astype(int)

# Drop original price column
df_house = df_house.drop(columns=["price"])

In [20]:
# Separate features and target variable
X = df_house.drop(columns=["price_class"]).values
y = df_house["price_class"].values

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [21]:
# Scale features to improve gradient descent convergence
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
def add_bias(X):
  """ Adds a bias term (column of 1s) to the feature matrix.

  Args:
    X (numpy.ndarray): Feature matrix of shape (n_samples, n_features)

  Returns:
    numpy.ndarray: Feature matrix with bias column added
  """

  # Create column of ones for intercept
  ones = np.ones((X.shape[0], 1))

  # Concatenate ones with original features
  return np.hstack((ones, X))

In [23]:
def sigmoid(z):
  """ Computes the sigmoid activation function.

  Args:
    z (numpy.ndarray or float): Linear input value

  Returns:
    numpy.ndarray or float: Probability values between 0 and 1
  """

  return 1 / (1 + np.exp(-z))

In [24]:
def cross_entropy_loss(X, y, theta):
  """ Computes cross-entropy loss for logistic regression.

  Args:
    X (numpy.ndarray): Feature matrix
    y (numpy.ndarray): Binary target vector
    theta (numpy.ndarray): Model parameters

  Returns:
    float: Cross-entropy loss value
  """

  # Add bias column
  X_b = add_bias(X)

  # Number of training samples
  n = len(y)

  # Predicted probabilities
  preds = sigmoid(X_b @ theta)

  # Prevent log(0)
  eps = 1e-15
  preds = np.clip(preds, eps, 1 - eps)

  # Compute cross-entropy loss
  loss = -(1/n) * np.sum(
      y*np.log(preds) + (1-y)*np.log(1-preds)
  )

  return loss

In [25]:
def logistic_gradient_descent(X, y, alpha, num_iters):
  """ Trains logistic regression using gradient descent.

  Args:
    X (numpy.ndarray): Feature matrix
    y (numpy.ndarray): Binary target vector
    alpha (float): Learning rate
    num_iters (int): Number of iterations

  Returns:
    tuple:
      numpy.ndarray: Learned parameter vector theta
      list: Cross-entropy loss values at each iteration
  """

  # Add bias column
  X_b = add_bias(X)

  # Number of samples and parameters
  n_samples, n_features = X_b.shape

  # Initialize parameters to zero
  theta = np.zeros(n_features)

  # List to store loss values
  losses = []

  # Gradient descent loop
  for i in range(num_iters):

      # Compute predicted probabilities
      preds = sigmoid(X_b @ theta)

      # Compute gradient of cross-entropy loss
      gradient = (1/n_samples) * (X_b.T @ (preds - y))

      # Update parameters
      theta = theta - alpha * gradient

      # Compute loss for monitoring convergence
      loss = cross_entropy_loss(X, y, theta)

      losses.append(loss)

  return theta, losses

In [26]:
# Learning rates to test
learning_rates = [0.001, 0.01, 0.1]

# Iteration checkpoints
iterations_list = [10, 50, 100]

results = []

# Run experiments
for alpha in learning_rates:

    for num_iters in iterations_list:

        # Train model
        theta, losses = logistic_gradient_descent(
            X_train_scaled,
            y_train,
            alpha,
            num_iters
        )

        # Get final loss
        loss = losses[-1]

        results.append([
            alpha,
            num_iters,
            loss
        ])

# Convert results to DataFrame
results_df = pd.DataFrame(
    results,
    columns=[
        "Learning Rate",
        "Iterations",
        "Cross Entropy Loss"
    ]
)

print(results_df)

   Learning Rate  Iterations  Cross Entropy Loss
0          0.001          10            0.689449
1          0.001          50            0.675429
2          0.001         100            0.659506
3          0.010          10            0.659362
4          0.010          50            0.574753
5          0.010         100            0.521163
6          0.100          10            0.518801
7          0.100          50            0.406377
8          0.100         100            0.374353


In [27]:
metrics_results = []

for alpha in learning_rates:

    theta, losses = logistic_gradient_descent(
        X_train_scaled,
        y_train,
        alpha,
        100
    )

    X_train_b = add_bias(X_train_scaled)
    X_test_b = add_bias(X_test_scaled)

    train_probs = sigmoid(X_train_b @ theta)
    test_probs = sigmoid(X_test_b @ theta)

    train_pred = (train_probs >= 0.5).astype(int)
    test_pred = (test_probs >= 0.5).astype(int)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)

    metrics_results.append([
        alpha,
        train_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

metrics_df = pd.DataFrame(
    metrics_results,
    columns=[
        "Learning Rate",
        "Train Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

print(metrics_df)

   Learning Rate  Train Accuracy  Test Accuracy  Precision    Recall  F1 Score
0          0.001        0.768999       0.780708   0.824515  0.719707  0.768555
1          0.010        0.788780       0.797132   0.844374  0.734339  0.785522
2          0.100        0.830480       0.836919   0.871988  0.794239  0.831299
